In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = os.environ.get("AGL_ROOT", ".")

LEVELS_PATH  = os.path.join(ROOT, "Code Outputs", "Gap Interpolation Outputs",
                            "Unified_Interpolated_Levels.xlsx")
CLIMATE_PATH = os.path.join(ROOT, "Code Outputs", "Climate Data Extraction Outputs",
                            "Lake_Climate_Monthly.xlsx")
CLIM06_PATH  = os.path.join(ROOT, "Code Outputs", "Climate EDA Outputs",
                            "CLIM_06_delta_leadlag.csv")
DMI_PATH     = os.path.join(ROOT, "Climate Indices", "dmi.csv")

OUTDIR = os.path.join(ROOT, "Code Outputs", "Climate EDA Outputs")

# canonical protocol
TRAIN_END = "2020-12-01"
WINDOW_START = "1995-06-01"
WINDOW_END = "2025-12-01"

MAX_LAG = 12          # CLIM_06 searches 0..12
PLOT_LAGS = range(-6, 13)


# Loading
def load_levels():
    df = pd.read_excel(LEVELS_PATH)
    df["Date"] = pd.to_datetime(df["Date"])
    level = df.pivot(index="Date", columns="Reservoir", values="Level_m").sort_index()
    return level.asfreq("MS")

def load_climate():
    df = pd.read_excel(CLIMATE_PATH)
    df["Date"] = pd.to_datetime(df["Date"])
    wide = df.pivot(index="Date", columns="Reservoir", values="water_balance_mm").sort_index()
    return wide.asfreq("MS")

def load_dmi():
    if not os.path.exists(DMI_PATH):
        return None
    d = pd.read_csv(DMI_PATH)
    d["Date"] = pd.to_datetime(d["Date"])

    dmi_col = d.columns[1]          # the value column, whatever NOAA has named it
    
    s = d.set_index("Date")[dmi_col].sort_index()
    return s.asfreq("MS")


def deseasonalise(s):
    """Subtract the calendar-month mean. Matches the CLIM_06 description."""
    s = s.dropna()
    if len(s) == 0:
        return s
    clim = s.groupby(s.index.month).transform("mean")
    return s - clim


def lag_corr(target, driver, lags):
    """corr(target[t], driver[t-k]) for each k. k>0 = driver LEADS target."""
    out = {}
    for k in lags:
        d = driver.shift(k)
        j = target.index.intersection(d.dropna().index)
        t = target.reindex(j).dropna()
        j = t.index.intersection(d.dropna().index)
        if len(j) < 24:
            out[k] = np.nan
            continue
        out[k] = float(np.corrcoef(t.reindex(j), d.reindex(j))[0, 1])
    return pd.Series(out)


def peak_of(curve, lags_allowed):
    c = curve.reindex(lags_allowed).dropna()
    if len(c) == 0:
        return np.nan, np.nan
    k = c.abs().idxmax()
    return int(k), float(c.loc[k])


def analyse(level, wb, dmi, start=None, end=None):
    """Return per-lake dict of curves and peaks for level (stock) and dLevel (flux)."""
    res = {}
    lakes = [c for c in level.columns if c in wb.columns]
    for lk in lakes:
        lv = level[lk].dropna()
        w = wb[lk].dropna()
        if start:
            lv, w = lv[lv.index >= start], w[w.index >= start]
        if end:
            lv, w = lv[lv.index <= end], w[w.index <= end]

        lv_a = deseasonalise(lv)              # level anomaly   (STOCK)
        dl_a = deseasonalise(lv.diff())       # dLevel anomaly  (FLUX)
        w_a = deseasonalise(w)                # water-balance anomaly

        lags = list(range(0, MAX_LAG + 1))
        cur_stock = lag_corr(lv_a, w_a, PLOT_LAGS)
        cur_flux = lag_corr(dl_a, w_a, PLOT_LAGS)

        k_s, r_s = peak_of(cur_stock, lags)
        k_f, r_f = peak_of(cur_flux, lags)

        entry = dict(curve_stock=cur_stock, curve_flux=cur_flux,
                     stock_lag=k_s, stock_r=r_s, flux_lag=k_f, flux_r=r_f,
                     lag0_r=float(cur_flux.get(0, np.nan)),
                     dl_a=dl_a, w_a=w_a)

        if dmi is not None:
            dmi_a = deseasonalise(dmi)
            cur_dmi = lag_corr(dl_a, dmi_a, PLOT_LAGS)
            k_d, r_d = peak_of(cur_dmi, lags)
            entry.update(curve_dmi=cur_dmi, dmi_lag=k_d, dmi_r=r_d)
        res[lk] = entry
    return res


# Figures
def figure_scatter(res, outpath):
    """7 scatter panels (WB at peak lag vs dLevel anomaly) + lag-curve panel."""
    lakes = sorted(res)
    fig, axes = plt.subplots(2, 4, figsize=(19, 9))
    axes = axes.flatten()

    for ax, lk in zip(axes, lakes):
        e = res[lk]
        k = e["flux_lag"]
        d = e["w_a"].shift(k)
        j = e["dl_a"].index.intersection(d.dropna().index)
        x = d.reindex(j).values
        y = e["dl_a"].reindex(j).values
        m = ~(np.isnan(x) | np.isnan(y))
        x, y = x[m], y[m]

        ax.scatter(x, y, s=11, alpha=.45, color="tab:blue", edgecolors="none")
        if len(x) > 2:
            b, a = np.polyfit(x, y, 1)
            xs = np.linspace(x.min(), x.max(), 50)
            ax.plot(xs, a + b * xs, color="tab:red", lw=2)
        ax.axhline(0, color="k", lw=.6); ax.axvline(0, color="k", lw=.6)
        ax.set_title(lk, fontweight="bold", fontsize=11)
        ax.set_xlabel(f"water-balance anomaly, lag {k} (mm)", fontsize=9)
        ax.set_ylabel("deseasonalised $\\Delta$level (m)", fontsize=9)
        ax.text(.03, .95, f"r = {e['flux_r']:.3f}\nn = {len(x)}",
                transform=ax.transAxes, va="top", fontsize=10, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.6", alpha=.9))

    ax = axes[7]
    for lk in lakes:
        c = res[lk]["curve_flux"]
        ax.plot(c.index, c.values, marker="o", ms=3, lw=1.3,
                label=lk.replace("Lake ", ""))
    ax.axvline(1, color="tab:red", ls="--", lw=1.6)
    ax.axhline(0, color="k", lw=.8)
    ax.set_title("Water balance $\\rightarrow$ $\\Delta$level: r by lag",
                 fontweight="bold", fontsize=11)
    ax.set_xlabel("lag k (months); climate leads $\\Delta$level", fontsize=9)
    ax.set_ylabel("correlation", fontsize=9)
    ax.legend(fontsize=7.5, ncol=2, loc="lower left")
    ax.text(1.15, ax.get_ylim()[1] * .92, "lag 1", color="tab:red", fontsize=9)

    fig.suptitle("Water-Balance Anomaly against Deseasonalised Monthly Level CHANGE",
                 fontweight="bold", fontsize=15, y=.995)
    fig.tight_layout()
    fig.savefig(outpath, dpi=800)
    plt.close(fig)
    print(f"  wrote {outpath}")


def figure_stock_vs_flux(res, outpath):
    """The CLIM_03 vs CLIM_06 contrast: r-vs-lag for level and dLevel."""
    lakes = sorted(res)
    fig, axes = plt.subplots(2, 4, figsize=(19, 8.5))
    axes = axes.flatten()
    # shared, adaptive y-limits so no peak is clipped
    allv = np.concatenate([np.concatenate([res[lk]["curve_stock"].values,
                                           res[lk]["curve_flux"].values])
                           for lk in lakes])
    allv = allv[~np.isnan(allv)]
    lo, hi = min(-.25, allv.min() - .08), max(.35, allv.max() + .12)
    for ax, lk in zip(axes, lakes):
        e = res[lk]
        ax.plot(e["curve_stock"].index, e["curve_stock"].values, marker="o", ms=3,
                lw=1.5, color="tab:orange", label="vs level (stock)")
        ax.plot(e["curve_flux"].index, e["curve_flux"].values, marker="o", ms=3,
                lw=1.8, color="tab:blue", label="vs $\\Delta$level (flux)")
        ax.axhline(0, color="k", lw=.8)
        ax.axvline(1, color="tab:red", ls="--", lw=1.2)
        ax.set_ylim(lo, hi)
        ax.set_title(lk, fontweight="bold", fontsize=11)
        ax.set_xlabel("lag k (months)", fontsize=9)
        ax.set_ylabel("correlation", fontsize=9)
        ax.text(.03, .95,
                f"stock: {e['stock_r']:+.3f} @ {e['stock_lag']}\n"
                f"flux:  {e['flux_r']:+.3f} @ {e['flux_lag']}",
                transform=ax.transAxes, va="top", fontsize=8.5, family="monospace",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.6", alpha=.9))
        if lk == lakes[0]:
            ax.legend(fontsize=8, loc="lower right")
    axes[7].axis("off")
    axes[7].text(.05, .5,
                 "Lake level is the time-integral of\nthe water balance.\n\n"
                 "Correlating climate against the LEVEL\n(a stock) gives weak, scattered\n"
                 "peaks at long lags.\n\n"
                 "Correlating against the LEVEL CHANGE\n(a flux) gives a sharp peak at\n"
                 "lag 1 for six of seven lakes.\n\n"
                 "This is why the exogenous frames\nare shifted by one month.",
                 fontsize=11, va="center")
    fig.suptitle("Stock versus Flux: why the water balance predicts $\\Delta$level, not level",
                 fontweight="bold", fontsize=15, y=.995)
    fig.tight_layout()
    fig.savefig(outpath, dpi=800)
    plt.close(fig)
    print(f"  wrote {outpath}")


# Main
def main():
    os.makedirs(OUTDIR, exist_ok=True)
    print("Loading levels ...")
    level = load_levels()
    print(f"  {level.shape[0]} months x {level.shape[1]} lakes")
    print("Loading climate ...")
    wb = load_climate()
    print(f"  {wb.shape[0]} months x {wb.shape[1]} lakes")
    dmi = load_dmi()
    print(f"  DMI: {'loaded' if dmi is not None else 'NOT FOUND - DMI check skipped'}")

    print("\nAnalysing FULL record (matches CLIM_06) ...")
    full = analyse(level, wb, dmi)
    print("Analysing TRAINING WINDOW only (leak check) ...")
    train = analyse(level, wb, dmi, start=WINDOW_START, end=TRAIN_END)

    # self-verification against CLIM_06
    print("\n" + "=" * 74)
    print("SELF-CHECK: recomputation vs CLIM_06_delta_leadlag.csv")
    print("=" * 74)
    ref = None
    if os.path.exists(CLIM06_PATH):
        ref = pd.read_csv(CLIM06_PATH).set_index("Lake")
    else:
        print(f"  CLIM_06 not found at {CLIM06_PATH} - skipping cross-check")

    rows = []
    n_ok = n_tot = 0
    for lk in sorted(full):
        e, t = full[lk], train[lk]
        row = dict(Lake=lk,
                   wb_peak_lag_full=e["flux_lag"], wb_peak_corr_full=round(e["flux_r"], 3),
                   wb_lag0_full=round(e["lag0_r"], 3),
                   wb_peak_lag_train=t["flux_lag"], wb_peak_corr_train=round(t["flux_r"], 3),
                   level_peak_lag_full=e["stock_lag"], level_peak_corr_full=round(e["stock_r"], 3))
        if "dmi_lag" in e:
            row.update(DMI_peak_lag_full=e["dmi_lag"], DMI_peak_corr_full=round(e["dmi_r"], 3),
                       DMI_peak_lag_train=t["dmi_lag"], DMI_peak_corr_train=round(t["dmi_r"], 3))
        if ref is not None and lk in ref.index:
            for mine, theirs in [("wb_peak_lag_full", "wb_peak_lag"),
                                 ("DMI_peak_lag_full", "DMI_peak_lag")]:
                if mine in row and theirs in ref.columns:
                    n_tot += 1
                    ok = int(row[mine]) == int(ref.loc[lk, theirs])
                    n_ok += ok
                    row[f"check_{theirs}"] = "PASS" if ok else \
                        f"FAIL (CLIM_06={int(ref.loc[lk, theirs])})"
            row["CLIM06_wb_peak_corr"] = ref.loc[lk, "wb_peak_corr"]
            row["CLIM06_DMI_peak_lag"] = ref.loc[lk, "DMI_peak_lag"]
        rows.append(row)

    stats = pd.DataFrame(rows)
    print(stats.to_string(index=False))
    
    if n_tot:
        wb_ok = sum(1 for r in rows if r.get("check_wb_peak_lag") == "PASS")
        print(f"\n  lag reproduction: {n_ok}/{n_tot} PASS "
              f"(water balance {wb_ok}/{len(rows)})")
        if wb_ok == len(rows):
            print("  Water-balance lags reproduce CLIM_06 exactly - the figures are sound.")

    # leak quantification
    print("\n" + "=" * 74)
    print("LEAK CHECK: do the selected lags change if the test period is excluded?")
    print("=" * 74)
    changed = []
    for lk in sorted(full):
        for key, name in [("flux_lag", "water balance"), ("dmi_lag", "DMI")]:
            if key in full[lk] and key in train[lk]:
                a, b = full[lk][key], train[lk][key]
                flag = "same" if a == b else f"CHANGED {a} -> {b}"
                if a != b:
                    changed.append((lk, name, a, b))
                print(f"  {lk:18s} {name:14s} full={a:>2}  train={b:>2}   {flag}")
    if not changed:
        print("\n  RESULT: no selected lag changes when the test period is excluded.")

    else:
        print(f"\n  RESULT: {len(changed)} lag(s) change. The leak is not immaterial;")

    stats.to_csv(os.path.join(OUTDIR, "CLIM_07_wb_dlevel_stats.csv"), index=False)
    print(f"\n  wrote {os.path.join(OUTDIR, 'CLIM_07_wb_dlevel_stats.csv')}")

    print("\nBuilding figures ...")
    figure_scatter(full, os.path.join(OUTDIR, "CLIM_07_wb_vs_dlevel.png"))
    figure_stock_vs_flux(full, os.path.join(OUTDIR, "CLIM_07_stock_vs_flux.png"))
    print("\nDone.")


if __name__ == "__main__":
    main()

Loading levels ...
  403 months x 7 lakes
Loading climate ...
  400 months x 7 lakes
  DMI: loaded

Analysing FULL record (matches CLIM_06) ...
Analysing TRAINING WINDOW only (leak check) ...

SELF-CHECK: recomputation vs CLIM_06_delta_leadlag.csv
           Lake  wb_peak_lag_full  wb_peak_corr_full  wb_lag0_full  wb_peak_lag_train  wb_peak_corr_train  level_peak_lag_full  level_peak_corr_full  DMI_peak_lag_full  DMI_peak_corr_full  DMI_peak_lag_train  DMI_peak_corr_train check_wb_peak_lag check_DMI_peak_lag  CLIM06_wb_peak_corr  CLIM06_DMI_peak_lag
    Lake Albert                 1              0.353         0.260                  1               0.353                    0                -0.098                  3               0.003                  10                0.002              PASS               PASS                0.351                    3
    Lake Edward                 0              0.293         0.293                  0               0.323                    3          